# Synthetic Kannada caller turns (#7)

Runs `training/generate_data.py`: labels are sampled from `data/generation/scenarios.json`, an instruction-tuned model writes each caller sentence in spoken Kannada, and only sentences that pass the consistency checks are kept.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2**, Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a write token).
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on. One T4 loads the model in 4-bit.
- Set `DATASET_REPO` below to the private dataset repo from #8.

Step 1 is a 40-row pilot per candidate model, so a Kannada reader can compare them before the long run.

In [ ]:
!git clone --depth 1 https://github.com/Pranav-0710/dhwani-desk.git
%cd dhwani-desk/training
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os

import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

DATASET_REPO = 'CHANGE-ME/dhwani-desk-data'  # private dataset repo from #8
FOUR_BIT = '' if torch.cuda.device_count() >= 2 else '--four-bit'
print(torch.cuda.device_count(), 'GPU(s)', FOUR_BIT or 'fp16')

## Step 1: pilot, 40 rows per candidate

Compare the pass rate and rejection reasons, then read the samples. The model that writes the most natural spoken Kannada wins, even at a lower pass rate.

In [ ]:
CANDIDATES = {'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct', 'qwen3-8b': 'Qwen/Qwen3-8B'}
for short, model in CANDIDATES.items():
    !python generate_data.py --n 40 --model {model} --out out/pilot-{short} {FOUR_BIT}

In [ ]:
import json

for short in CANDIDATES:
    report = json.load(open(f'out/pilot-{short}/report.json', encoding='utf-8'))
    print()
    print(f"== {short}: kept {report['kept']}/{report['requested']}, "
          f"first-try pass {report['first_attempt_pass_rate']:.0%}, {report['rejection_reasons']}")
    rows = [json.loads(line) for line in open(f'out/pilot-{short}/train.jsonl', encoding='utf-8')]
    for row in rows[:12]:
        print(f"{row['labels']['department']:>15} | {row['text']} | {row['gloss']}")

## Step 2: full run and upload

Set `MODEL` to the pilot winner. 2,500 rows take roughly 1–2 hours on T4 x2. The run writes `train.jsonl`, `rejected.jsonl`, `report.json` and a dataset card, then uploads them to `DATASET_REPO`. Versioning and the 10% hand check are #12.

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
!python generate_data.py --n 2500 --model {MODEL} --out out/train {FOUR_BIT} --push-repo {DATASET_REPO}